# Orchestrierungsmethoden

In [16]:
import json
import os
from openai import OpenAI
from dotenv import load_dotenv

## Metadata

In [17]:
tools = [
    {
        "name": "get_data",
        "description": "Retrieve time series energy data for a specific metering point and time range.",
        "parameters": {
            "type": "object",
            "properties": {
                "meter_id": {
                    "type": "string",
                    "description": "Unique identifier of the metering point"
                },
                "start": {
                    "type": "string",
                    "description": "Start date in ISO format (YYYY-MM-DD)"
                },
                "end": {
                    "type": "string",
                    "description": "End date in ISO format (exclusive)"
                },
                "forecast": {
                    "type": "boolean",
                    "description": "Whether to use forecast or actual data"
                }
            },
            "required": ["meter_id"]
        }
    },
    {
        "name": "create_plot",
        "description": "Create a time series plot for one or multiple metering points.",
        "parameters": {
            "type": "object",
            "properties": {
                "meter_ids": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "List of metering point IDs"
                },
                "start": {
                    "type": "string",
                    "description": "Start date (YYYY-MM-DD)"
                },
                "end": {
                    "type": "string",
                    "description": "End date (exclusive)"
                },
                "forecast": {
                    "type": "boolean",
                    "description": "Use forecast or actual data"
                }
            },
            "required": ["meter_ids"]
        }
    },
    {
        "name": "get_statistics",
        "description": "Calculate summary statistics such as mean, min, and max for a metering point.",
        "parameters": {
            "type": "object",
            "properties": {
                "meter_id": {
                    "type": "string"
                },
                "start": {
                    "type": "string"
                },
                "end": {
                    "type": "string"
                },
                "forecast": {
                    "type": "boolean"
                }
            },
            "required": ["meter_id"]
        }
    },
    {
        "name": "get_data_sample",
        "description": "Return a small sample of time series data for interpretation by the model.",
        "parameters": {
            "type": "object",
            "properties": {
                "meter_id": {"type": "string"},
                "start": {"type": "string"},
                "end": {"type": "string"},
                "forecast": {"type": "boolean"},
                "max_points": {
                    "type": "integer",
                    "description": "Maximum number of data points to return"
                }
            },
            "required": ["meter_id"]
        }
    }
]

In [25]:
metadata = {'metadata': [
    {'meter_id': 'mp_1', 'location': 'Wohnhaus', 'direction': 'consumption'},
    {'meter_id': 'mp_2', 'location': 'Wohnhaus', 'direction': 'generation'},
    {'meter_id': 'mp_3', 'location': 'Firma', 'direction': 'consumption'},
    {'meter_id': 'mp_4', 'location': 'Firma', 'direction': 'generation'}
    ],
'unit': 'kWh'}

metadata_str = json.dumps(metadata, indent=2)
print(metadata)

{'metadata': [{'meter_id': 'mp_1', 'location': 'Wohnhaus', 'direction': 'consumption'}, {'meter_id': 'mp_2', 'location': 'Wohnhaus', 'direction': 'generation'}, {'meter_id': 'mp_3', 'location': 'Firma', 'direction': 'consumption'}, {'meter_id': 'mp_4', 'location': 'Firma', 'direction': 'generation'}], 'unit': 'kWh'}


## LLM Call

In [19]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI()

def call_llm(user_input: str, systeminformation: str, client=client, messages=[]):
    response = client.chat.completions.create(
        model="gpt-5.4",
        # model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": systeminformation},
            *messages,
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content

## Deterministisch

## Plan-basiert

In [29]:
SYSTEMINFORMATION_2 = """
You are an AI agent that plans how to solve data analysis tasks using tools.

Available metering points:
{'metadata': [
    {'meter_id': 'mp_1', 'location': 'Wohnhaus', 'direction': 'consumption'},
    {'meter_id': 'mp_2', 'location': 'Wohnhaus', 'direction': 'generation'},
    {'meter_id': 'mp_3', 'location': 'Firma', 'direction': 'consumption'},
    {'meter_id': 'mp_4', 'location': 'Firma', 'direction': 'generation'}
    ],
'unit': 'kWh'}

You do NOT execute tools. You ONLY create a plan.

Available tools:

1. get_data(meter_id, start, end, forecast)
   - Retrieves time series data for a metering point

2. create_plot(meter_ids, start, end, forecast)
   - Creates a plot for one or multiple metering points

3. get_statistics(meter_id, start, end, forecast)
   - Computes summary statistics (mean, min, max)

---

Planning rules:

1. Return ONLY valid JSON. No explanations.
2. The output must follow this structure:

{
  "plan": [
    {
      "id": "step1",
      "tool": "tool_name",
      "args": { ... }
    }
  ]
}

3. Each step must have a unique "id" (e.g. step1, step2, step3).

4. If a step depends on the output of previous steps, reference them using:

"input_steps": ["step1", "step2"]

5. Do NOT pass raw data between steps. Only reference previous step IDs.

6. Prefer minimal plans:
   - Do not call get_data if another tool can retrieve the data internally
   - Only include necessary steps

7. Time handling:
   - If the user says "yesterday", use:
     start = YYYY-MM-DD of yesterday
     end = YYYY-MM-DD of today (exclusive)

8. Assume timestamps follow [start, end) (end is exclusive).

9. Always include required arguments.

10. If multiple metering points are involved, handle them explicitly.

---

Your task is to generate a correct and efficient execution plan.
"""

In [31]:
user_input = "Vergleiche meine Verbräuche."
plan = call_llm(user_input, SYSTEMINFORMATION_2)
plan

'{"plan":[{"id":"step1","tool":"create_plot","args":{"meter_ids":["mp_1","mp_3"],"start":"2026-04-13","end":"2026-04-14","forecast":false}},{"id":"step2","tool":"get_statistics","args":{"meter_id":"mp_1","start":"2026-04-13","end":"2026-04-14","forecast":false}},{"id":"step3","tool":"get_statistics","args":{"meter_id":"mp_3","start":"2026-04-13","end":"2026-04-14","forecast":false}}]}'

In [33]:
print(plan)

{"plan":[{"id":"step1","tool":"create_plot","args":{"meter_ids":["mp_1","mp_3"],"start":"2026-04-13","end":"2026-04-14","forecast":false}},{"id":"step2","tool":"get_statistics","args":{"meter_id":"mp_1","start":"2026-04-13","end":"2026-04-14","forecast":false}},{"id":"step3","tool":"get_statistics","args":{"meter_id":"mp_3","start":"2026-04-13","end":"2026-04-14","forecast":false}}]}


## Iterativ